In [1]:
!pip install mitsuba

In [2]:
import mitsuba as mi
import matplotlib.pyplot as plt
import numpy as np
import math

In [3]:
mi.set_variant('cuda_ad_spectral')

In [4]:
!wget https://raw.githubusercontent.com/gnomeby/canvas3D/refs/heads/master/teapot.ply

--2026-11-14 00:32:44--  https://raw.githubusercontent.com/gnomeby/canvas3D/refs/heads/master/teapot.ply
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 63132 (62K) [text/plain]
Saving to: ‘teapot.ply.17’

teapot.ply.17       100%[===================>]  61.65K  --.-KB/s    in 0.05s   

2026-11-14 00:32:44 (1.11 MB/s) - ‘teapot.ply.17’ saved [63132/63132]



In [5]:
scene = mi.load_dict({
    'type': 'scene',
    'integrator': {'type': 'path'},

    'light': {
        'type': 'directional',
        'direction': [-0.5, -0.7, -1.0],
        'irradiance': {'type': 'rgb', 'value': [3.0, 3.0, 3.0]},
    },

    'fill': {
        'type': 'directional',
        'direction': [0.6, 0.3, 0.8],
        'irradiance': {'type': 'rgb', 'value': [0.8, 0.8, 0.8]},
    },

    'teapot': {
        'type': 'ply',
        'filename': 'teapot.ply',
        'to_world': mi.ScalarTransform4f().scale(0.5).translate([0, 0, 1.5]).rotate(axis=[1, 0, 0], angle=-90),
        'bsdf': {'type': 'diffuse', 'reflectance': {'type': 'rgb', 'value': [0.6, 0.6, 0.6]}}
    },
})

In [6]:
def spherical_positions(width: int, height: int, N: int, x: float, y: float, z: float, radius: float):
    sensors = []
    phi = np.pi * (3. - np.sqrt(5.))
    
    target = mi.ScalarPoint3f(x, y, z)

    for i in range(N):
        y_coord = 1 - (i / float(N - 1)) * 2
        radius_at_y = np.sqrt(1 - y_coord * y_coord)
        theta = phi * i
        
        px = np.cos(theta) * radius_at_y
        py = y_coord
        pz = np.sin(theta) * radius_at_y
        
        cam_pos = mi.ScalarPoint3f(
            x + px * radius,
            y + py * radius,
            z + pz * radius
        )
        up = (0.0, 0.0, 1.0) if abs(py) > 0.999 else (0.0, 1.0, 0.0)
        
        sensor = {
            "type": "perspective",
            "fov": 70,
            "to_world": mi.ScalarTransform4f().look_at(cam_pos, target, mi.ScalarPoint3f(*up)),
            'film': {
                'type': 'hdrfilm',
                'width': width,
                'height': height,
            }
        }
        sensors.append(sensor)
    return sensors

def use_spectrum(sensors: list):
    wavelengths = list(range(450, 651, 10))
    res = []

    for rgb_sensor in sensors:
        width, height = rgb_sensor['film']['width'], rgb_sensor['film']['height']
        wavelength_sensors = dict()
        wavelength_sensors['rgb'] = rgb_sensor

        for wavelength in wavelengths:
            film = {
                'type': 'specfilm',
                'width': width,
                'height': height,
                'channel1': {
                    'type': 'irregular',
                    'wavelengths': f'{wavelength}, {wavelength + 1}',
                    'values': '1.0, 1.0',
                },
            }
            wavelength_sensors[wavelength] = {**rgb_sensor, 'film': film}
        res.append(wavelength_sensors)
    return res

In [7]:
rgb_cfgs = spherical_positions(640, 480, 50, 0, 0, 0, 5)

spectral_cfgs = use_spectrum(rgb_cfgs)

sensors = []
for group in spectral_cfgs:
    loaded_group = {k: mi.load_dict(v, parallel=False) for k, v in group.items()}
    sensors.append(loaded_group)

In [ ]:
global images
if globals().get('images'):
    print("Images is not empty. Type 'Y' to continue")
    if input().strip().upper() == 'Y':
        images = list()
if not globals().get('images'):
    images = list()
    print('rendering started')
    for i, sensorWavelength in enumerate(sensors):
        spectrum = dict()
        for wavelength, sensor in sensorWavelength.items():
            spectrum[wavelength] = mi.render(scene, sensor=sensor, spp=256)
        images.append(spectrum)
        print(f'{i+1}/{len(sensors)}')
    print('rendering finished')

rendering started
1/50
2/50
3/50
4/50
5/50
6/50
7/50
8/50


In [ ]:
def split_train_val(images, val_every=8):
    train_images = []
    val_images = []
    train_indices = []
    val_indices = []

    for i, spectrum in enumerate(images):
        if i % val_every == 0:
            val_images.append(spectrum)
            val_indices.append(i)
        else:
            train_images.append(spectrum)
            train_indices.append(i)

    return train_images, val_images, train_indices, val_indices


In [ ]:
train, val, train_idx, val_idx = split_train_val(images)


In [ ]:
import json
import os

import numpy as np
from PIL import Image


def save_dataset(images, sensor_cfgs, train_idx, val_idx, base_dir, wavelengths):
    width = sensor_cfgs[0]["film"]["width"]
    height = sensor_cfgs[0]["film"]["height"]
    fov = float(sensor_cfgs[0]["fov"])
    fl = 0.5 * width / math.tan(math.radians(fov) * 0.5)
    meta = {
        "camera_angle_x": math.radians(fov),
        "fl_x": fl,
        "fl_y": fl,
        "cx": 0.5 * width,
        "cy": 0.5 * height,
        "w": width,
        "h": height,
        "camera_model": "OPENCV",
        "k1": 0.0,
        "k2": 0.0,
        "p1": 0.0,
        "p2": 0.0,
        "wavelengths": [int(w) for w in wavelengths],
    }

    os.makedirs(os.path.join(base_dir, "train"), exist_ok=True)
    os.makedirs(os.path.join(base_dir, "eval"), exist_ok=True)

    val_idx = set(val_idx)
    frames = []

    for idx, (image, cfg) in enumerate(zip(images, sensor_cfgs)):
        split = "eval" if idx in val_idx else "train"
        name = f"r_{idx}"

        rgb = np.clip(np.asarray(image["rgb"])[..., :3], 0.0, 1.0)
        rgb = np.where(rgb < 0.0031308, 12.92 * rgb, 1.055 * np.power(rgb, 1 / 2.4) - 0.055)
        Image.fromarray((rgb * 255).round().astype(np.uint8)).save(
            os.path.join(base_dir, split, f"{name}.png")
        )

        cube = np.zeros((height, width, len(wavelengths)), dtype=np.float32)
        for i, wavelength in enumerate(wavelengths):
            band = np.asarray(image[wavelength])
            if band.ndim == 3:
                band = band[..., 0]
            cube[..., i] = np.clip(band, 0.0, 1.0)
        np.save(os.path.join(base_dir, split, f"{name}.npy"), cube)

        c2w = np.array(cfg["to_world"].matrix, dtype=np.float64)
        c2w[:3, 0] *= -1
        c2w[:3, 2] *= -1

        frames.append({
            "file_path": f"{split}/{name}.png",
            "rotation": 0.0,
            "transform_matrix": c2w.tolist(),
            "hyperspectral_file_path": f"{split}/{name}.npy",
        })

    train_frames = [frames[i] for i in train_idx]
    val_frames = [frames[i] for i in val_idx]

    for name, split_frames in (("train", train_frames), ("val", val_frames), ("test", frames)):
        with open(os.path.join(base_dir, f"transforms_{name}.json"), "w") as f:
            json.dump({**meta, "frames": split_frames}, f, indent=4)

    with open(os.path.join(base_dir, "transforms.json"), "w") as f:
        json.dump({**meta, "frames": frames}, f, indent=4)


In [ ]:
wavelengths = sorted(k for k in images[0] if k != "rgb")

save_dataset(images, rgb_cfgs, train_idx, val_idx, "data/my_scene", wavelengths)


In [ ]:
wavelengths = sorted(k for k in images[0] if k != "rgb")

save_dataset(images, rgb_cfgs, train_idx, val_idx, "data/my_scene_raw", wavelengths)


In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt

def show_images(images: dict, gamma: float = 2.2, cols: int = 4, cmap=None):
    items = list(images.items())
    n = len(items)
    cols = min(cols, n)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows), squeeze=False)

    for ax, (title, image) in zip(axes.flat, items):
        img = np.asarray(image, dtype=np.float64)

        img = np.clip(img, 0.0, 1.0)

        img_g = np.power(img, 1.0 / gamma)

        ax.imshow(img_g, cmap=cmap, vmin=0.0, vmax=1.0)
        ax.set_title(title)
        ax.axis("off")

    for ax in axes.flat[n:]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()

In [ ]:
show_images(images[5])